# Heart Failure Prediction — testes de hiperparâmetros
**Algoritmos:** Árvore de decisão, rede neural MLP, Random Forest e K-means (mais um baseline trivial para referência).

Faça upload do arquivo `heart.csv` da [base do Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction). Execute as células de cima para baixo.

**Método**
- Treino/teste estratificados (80/20). O teste só é usado depois da seleção dos hiperparâmetros.
- Zeros fisiologicamente impossíveis (`Cholesterol`, `RestingBP`) são tratados como ausentes e imputados pela mediana **dentro do pipeline** (ajustados apenas no treino de cada fold).
- Busca em grade com validação cruzada estratificada repetida (5 folds × 2 repetições), métrica F1 da classe 1.
- Padronização e one-hot também ajustados dentro de cada fold (sem vazamento).
- Incerteza do teste estimada por bootstrap (intervalo de 95% do F1).
- K-means é exploratório: k é escolhido por Silhouette e estabilidade, sem usar `HeartDisease`; os rótulos entram só na interpretação final.

In [ ]:
import os, json, zipfile, unicodedata, warnings
from pathlib import Path
os.environ['PYTHONWARNINGS'] = 'ignore'   # propaga aos workers do n_jobs=-1
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.model_selection import (train_test_split, GridSearchCV, RepeatedStratifiedKFold,
                                     ParameterGrid, cross_val_score)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay,
                             silhouette_score, adjusted_rand_score)

SEED = 42
print('scikit-learn', sklearn.__version__)

try:
    from google.colab import files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if Path('heart.csv').exists():
    print('Usando heart.csv já presente no ambiente.')
elif EM_COLAB:
    uploaded = files.upload()
    csvs = [n for n in uploaded if n.lower().endswith('.csv')]
    if not csvs:
        raise ValueError('Envie o arquivo heart.csv do Kaggle.')
    Path(csvs[0]).rename('heart.csv')   # aceita nomes como "heart (1).csv"
else:
    raise FileNotFoundError('Coloque heart.csv na pasta do notebook.')

## Carga e limpeza dos dados
No `heart.csv` original não há `NaN`, mas `Cholesterol` tem muitos zeros (dado faltante codificado como 0) e `RestingBP` tem um. Aqui eles viram `NaN` e são imputados no pipeline.

In [ ]:
df = pd.read_csv('heart.csv')
assert 'HeartDisease' in df.columns, 'Coluna HeartDisease não encontrada.'
print('Formato:', df.shape)
display(df.head())
print('Classes:\n', df['HeartDisease'].value_counts().to_string())
print('Valores ausentes (NaN):', int(df.isna().sum().sum()))
print('Linhas duplicadas:', int(df.duplicated().sum()))

# zeros impossíveis -> NaN (regra determinística, sem vazamento; imputação fica no pipeline)
for c in ['Cholesterol', 'RestingBP']:
    if c in df.columns:
        n0 = int((df[c] == 0).sum())
        print(f'{c}: {n0} zeros convertidos em NaN')
        df[c] = df[c].replace(0, np.nan)

## Preparação dos dados

In [ ]:
X = df.drop(columns='HeartDisease')
y = df['HeartDisease']
cat = X.select_dtypes(include=['object', 'category']).columns.tolist()
num = X.select_dtypes(include='number').columns.tolist()

def make_ohe():
    # compatível com scikit-learn >= 1.2 (sparse_output) e versões anteriores (sparse)
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num),
    ('cat', make_ohe(), cat)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
print('Treino:', len(X_train), '| Teste:', len(X_test))
print('Proporção da classe 1 -> treino: %.3f | teste: %.3f' % (y_train.mean(), y_test.mean()))

## Buscas de hiperparâmetros (F1 da classe 1)
A validação cruzada (treino apenas) escolhe os hiperparâmetros. O teste fica reservado até o fim da seleção.

In [ ]:
configuracoes = {
    'Árvore de decisão': (
        DecisionTreeClassifier(random_state=SEED),
        {'modelo__criterion': ['gini', 'entropy'],
         'modelo__max_depth': [3, 5, 7, 10],
         'modelo__min_samples_split': [2, 5, 10],
         'modelo__min_samples_leaf': [1, 3, 5]}),
    'Rede neural MLP': (
        MLPClassifier(solver='adam', max_iter=2000, random_state=SEED),
        {'modelo__hidden_layer_sizes': [(16,), (32, 16), (64, 32)],
         'modelo__activation': ['relu', 'tanh'],
         'modelo__alpha': [0.0001, 0.001, 0.01],
         'modelo__learning_rate_init': [0.001, 0.01]}),
    'Random Forest': (
        RandomForestClassifier(random_state=SEED, n_jobs=1),
        {'modelo__n_estimators': [100, 200, 300],
         'modelo__max_depth': [5, 10, None],
         'modelo__min_samples_leaf': [1, 2, 4]})
}

buscas = {}
for nome, (estimador, grade) in configuracoes.items():
    print('\n===', nome, '===')
    pipe = Pipeline([('prep', prep), ('modelo', estimador)])
    busca = GridSearchCV(pipe, grade, cv=cv, scoring='f1', n_jobs=-1,
                         refit=True, return_train_score=True, error_score='raise')
    busca.fit(X_train, y_train)
    buscas[nome] = busca
    i = busca.best_index_
    print('Combinações:', len(ParameterGrid(grade)), '| fits:', len(ParameterGrid(grade)) * cv.get_n_splits())
    print('Melhor F1 (validação): %.4f ± %.4f' % (busca.best_score_, busca.cv_results_['std_test_score'][i]))
    print('F1 no treino da CV:    %.4f' % busca.cv_results_['mean_train_score'][i])
    print('Melhores parâmetros:', busca.best_params_)

# 5 melhores configurações de cada algoritmo: mostra se a escolha é robusta ou se há "empate"
for nome, busca in buscas.items():
    r = pd.DataFrame(busca.cv_results_).sort_values('rank_test_score').head(5)
    print('\nTop 5 —', nome)
    display(r[['params', 'mean_test_score', 'std_test_score', 'mean_train_score']].round(4))

## Comparação no teste reservado

In [ ]:
def bootstrap_f1(y_true, pred, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        vals.append(f1_score(y_true[idx], pred[idx], zero_division=0))
    return np.percentile(vals, [2.5, 97.5])

linhas = []
fig, axes = plt.subplots(1, len(buscas), figsize=(5 * len(buscas), 4))
for ax, (nome, busca) in zip(np.atleast_1d(axes), buscas.items()):
    pred = busca.predict(X_test)
    i = busca.best_index_
    lo, hi = bootstrap_f1(y_test, pred)
    linhas.append({'Algoritmo': nome,
                   'F1_CV': busca.best_score_,
                   'F1_CV_desvio': busca.cv_results_['std_test_score'][i],
                   'F1_treino_CV': busca.cv_results_['mean_train_score'][i],
                   'Acuracia_teste': accuracy_score(y_test, pred),
                   'Precisao_classe1': precision_score(y_test, pred, zero_division=0),
                   'Recall_classe1': recall_score(y_test, pred, zero_division=0),
                   'F1_teste': f1_score(y_test, pred, zero_division=0),
                   'F1_teste_IC95_inf': lo, 'F1_teste_IC95_sup': hi})
    print('\n', nome)
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title(nome)
plt.tight_layout()
plt.show()

# baseline trivial (sempre a classe majoritária) para dar referência aos números
dummy = Pipeline([('prep', prep), ('modelo', DummyClassifier(strategy='most_frequent'))])
f1_cv_dummy = cross_val_score(dummy, X_train, y_train, cv=cv, scoring='f1').mean()
pred_d = dummy.fit(X_train, y_train).predict(X_test)
lo, hi = bootstrap_f1(y_test, pred_d)
linhas.append({'Algoritmo': 'Baseline (classe majoritária)', 'F1_CV': f1_cv_dummy,
               'F1_CV_desvio': np.nan, 'F1_treino_CV': np.nan,
               'Acuracia_teste': accuracy_score(y_test, pred_d),
               'Precisao_classe1': precision_score(y_test, pred_d, zero_division=0),
               'Recall_classe1': recall_score(y_test, pred_d, zero_division=0),
               'F1_teste': f1_score(y_test, pred_d, zero_division=0),
               'F1_teste_IC95_inf': lo, 'F1_teste_IC95_sup': hi})

comparacao = pd.DataFrame(linhas).sort_values('F1_CV', ascending=False).reset_index(drop=True)
display(comparacao.round(4))

so_modelos = comparacao[~comparacao['Algoritmo'].str.startswith('Baseline')]
print('Escolhido pela validação cruzada:', so_modelos.iloc[0]['Algoritmo'])
print('Melhor F1 observado no teste (descritivo):',
      so_modelos.sort_values('F1_teste', ascending=False).iloc[0]['Algoritmo'])
print('Atenção: com %d amostras de teste, se os IC95%% se sobrepõem, as diferenças não são conclusivas.' % len(y_test))

## K-means: análise não supervisionada
- O pré-processamento é **clonado** (`clone(prep)`) para não reaproveitar o ajuste usado nos classificadores.
- O número de clusters é escolhido por **Silhouette** e **estabilidade** (ARI entre execuções com sementes diferentes), sem olhar `HeartDisease`.
- `n_init` e `max_iter` ficam fixos: variá-los quase não muda o resultado quando o algoritmo já converge.
- Os rótulos reais só aparecem depois, para interpretação (ARI e taxa de doença por cluster). O ARI não seleciona nada.

In [ ]:
prep_k = clone(prep)
X_k_train = prep_k.fit_transform(X_train)   # ajustado só no treino
X_k_test = prep_k.transform(X_test)

registros, modelos_k = [], {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=20, max_iter=500, random_state=SEED)
    g = km.fit_predict(X_k_train)
    g2 = KMeans(n_clusters=k, n_init=20, max_iter=500, random_state=SEED + 1).fit_predict(X_k_train)
    modelos_k[k] = (km, g)
    registros.append({'k': k, 'Inercia': km.inertia_,
                      'Silhouette': silhouette_score(X_k_train, g),
                      'Estabilidade_ARI': adjusted_rand_score(g, g2)})
kresult = pd.DataFrame(registros)
display(kresult.round(4))

k_best = int(kresult.sort_values('Silhouette', ascending=False).iloc[0]['k'])
print('k escolhido por Silhouette:', k_best)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(kresult['k'], kresult['Inercia'], marker='o'); ax[0].set_title('Cotovelo (inércia)'); ax[0].set_xlabel('k')
ax[1].plot(kresult['k'], kresult['Silhouette'], marker='o', color='tab:orange'); ax[1].set_title('Silhouette'); ax[1].set_xlabel('k')
plt.tight_layout(); plt.show()

In [ ]:
# Interpretação (rótulos usados apenas aqui, depois da escolha de k)
ari_tab = pd.DataFrame([{'k': k, 'ARI_treino_vs_HeartDisease': adjusted_rand_score(y_train, g)}
                        for k, (km, g) in modelos_k.items()])
kresult = kresult.merge(ari_tab, on='k')
display(kresult.round(4))

km_final, g_train = modelos_k[k_best]
g_test = km_final.predict(X_k_test)
print(f'k={k_best} | Silhouette no teste: {silhouette_score(X_k_test, g_test):.4f} '
      f'| ARI no teste: {adjusted_rand_score(y_test, g_test):.4f}')
perfil = (pd.DataFrame({'cluster': g_train, 'HeartDisease': y_train.values})
            .groupby('cluster')['HeartDisease'].agg(['count', 'mean']).rename(
                columns={'count': 'n', 'mean': 'taxa_HeartDisease'}))
display(perfil.round(3))

# projeção 2D (PCA) só para visualização
Z = PCA(n_components=2, random_state=SEED).fit_transform(X_k_train)
plt.figure(figsize=(5.5, 4.5))
plt.scatter(Z[:, 0], Z[:, 1], c=g_train, cmap='viridis', s=12)
plt.title(f'K-means (k={k_best}) projetado em PCA'); plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

## Exportar resultados
Salva as métricas, as variações de busca e os melhores hiperparâmetros para uso no relatório.

In [ ]:
def slugify(txt):
    s = unicodedata.normalize('NFKD', txt).encode('ascii', 'ignore').decode()
    return s.lower().replace(' ', '_')

arquivos = ['comparacao_classificadores.csv', 'variacoes_kmeans.csv', 'melhores_hiperparametros.json']
comparacao.to_csv('comparacao_classificadores.csv', index=False)
kresult.to_csv('variacoes_kmeans.csv', index=False)

parametros = {}
for nome, busca in buscas.items():
    nome_arq = f'variacoes_{slugify(nome)}.csv'
    pd.DataFrame(busca.cv_results_).to_csv(nome_arq, index=False)
    arquivos.append(nome_arq)
    parametros[nome] = {k.replace('modelo__', ''): v for k, v in busca.best_params_.items()}
parametros['K-means'] = {'n_clusters': k_best, 'n_init': 20, 'max_iter': 500}

with open('melhores_hiperparametros.json', 'w', encoding='utf-8') as f:
    json.dump(parametros, f, ensure_ascii=False, indent=2, default=str)

with zipfile.ZipFile('resultados.zip', 'w') as z:
    for a in arquivos:
        z.write(a)
print('Criados:', ', '.join(arquivos), '+ resultados.zip')

if EM_COLAB:
    files.download('resultados.zip')